# 00 — Generate synthetic data (single scenario, illustrative)

**Purpose.** Show what the synthetic generator produces, so the rest of the study rests on
something you have seen rather than something you have taken on trust. This notebook covers
one `(p, kappa)` scenario only; the full grid sweep lives in `src/experiment.py`
(see notebook 02).

**Inputs.**

- `config.SIGMA` (= 1.0) and `config.BACKBONE_B` (= 10.0), from `src/config.py`.
- Illustrative parameters set inline in this notebook: `p = 0.1`, `kappa = 4.0`,
  `days = 40`, `seed = 999`. These are teaching values chosen for legibility, not results —
  nothing in the paper depends on them.

**Outputs.** None written to disk. Everything here is displayed inline.

**Process.**

1. Put `src/` on the path and import `config` and `synth`.
2. Generate 40 days for one scenario and inspect the returned frame.
3. Plot the observed load `l` to see what the estimator will actually be given.

## The model, in one line

At a fixed time of day, across `D` days:

$$L = B + Z A + \varepsilon, \qquad Z \sim \text{Bernoulli}(p), \qquad \varepsilon \sim N(0, \sigma^2), \qquad \kappa = A/\sigma$$

So on most days the meter reads the ordinary backbone plus noise, and on an *event* day —
an EV charging, a pool pump running — it also reads `A` on top. The whole study is the
question of how to recover `B` when you only ever see `L`.

## Setup

The notebooks are thin wrappers over `src/`, so each begins by putting that folder on the
import path. Nothing is duplicated here: `config` supplies every constant and `synth`
supplies the generator, exactly as `experiment.py` uses them.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import config
import synth

## Generate one illustrative scenario

Forty days rather than the study's 365, because the point here is a readable plot rather
than a result — a year of points is an unreadable smear at this figure size. `kappa = 4.0`
puts the event size at four times the day-to-day noise, well clear of the identifiability
bar of 2.83, so events are visible by eye; the hard cases come later. `p = 0.1` means about
one day in ten carries an event.

The returned frame keeps the components **separated**: `b_true`, `f_true` and `eps`
alongside the observed `l`. That separation is what makes evaluation possible — but note
that only `l` is ever passed to an estimator. The ground-truth columns exist to score
against, never to estimate from.

Also worth noticing: `b_true` is a constant. At a single fixed timestamp the backbone does
not vary across days by construction, which is the simplification the whole study rests on.

In [ ]:
days_df = synth.generate_days(
    p=0.1, kappa=4.0, sigma=config.SIGMA, backbone_b=config.BACKBONE_B, days=40, seed=999,
)
days_df.head()

## Plot the observed load

This is the estimator's entire view of the world: one number per day, with no label saying
which days carried an event. The backbone sits at 10 kW and the event days stand about
4 kW above it, so at `kappa = 4` they read as obvious upward excursions.

Keep this picture in mind for notebook 01, where the same plot at `kappa = 1` becomes
indistinguishable from noise — that is the regime the recoverability map exists to
characterise.

In [ ]:
days_df.plot(x="day", y="l", figsize=(8, 3), title="Illustrative observed load L")

## Conclusion

The generator does what the model says: `l` is `b_true + f_true + eps`, the three
components are retained separately for evaluation, and at `kappa = 4` roughly one day in
ten sits visibly above the backbone.

The task the rest of the study addresses is now concrete. Given only the `l` column, and
without being told which days were events, recover the level of `b_true`. Notebook 01 walks
through the three ways of doing that.